# Lab C10.1–C10.3 · Klucze, grupy i złączenia

Przed danym etapem przeczytaj wskazany wykład. Notebooki uruchamiaj ze wspólnego folderu `labs`, zgodnie z głównym README. Pracuj we własnej kopii repozytorium.

Przewiduj wyniki przed wykonaniem kodu. Nie musisz zapisywać odpowiedzi na pytania do przemyślenia. Sprawdzenia pustych starterów celowo nie przechodzą.

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3, 12), "Wybierz kernel Pythona 3.12 z głównego README."
lab_dir = Path.cwd()
assert (lab_dir / "lab-c10-1-c10-3.ipynb").is_file(), "Uruchom Jupyter z folderu labs."
print("Folder:", lab_dir)

In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise AssertionError("Brakuje bibliotek sekcji C. Dodaj je w practice i uruchom Jupyter przez --project ../practice, zgodnie z głównym README.") from error
print("pandas", pd.__version__, "| numpy", np.__version__)

## Panel trzech produktów

Gotowa komórka wczytuje `data/c-panel-sales.csv`: syntetyczną sprzedaż trzech produktów z dni 1–6. Wiersze są celowo w przypadkowej kolejności. Przewidź: ile wierszy powinno mieć 3 produkty × 6 dni? Porównaj z kształtem i zastanów się, czy sama liczba wierszy wystarcza, by uznać dane za kompletne.

In [ ]:
panel_path = lab_dir / "data" / "c-panel-sales.csv"
assert panel_path.is_file(), "Brakuje data/c-panel-sales.csv w kopii repozytorium."
panel = pd.read_csv(panel_path)
panel_before = panel.copy()
print(panel.dtypes)
print("Kształt:", panel.shape)
print(panel.sort_values(["item", "day"]))

## C10.1 · Braki i powtórzenia klucza

Po wykładzie C10.1 kluczem obserwacji jest para `item` i `day`. W `missing_sale` policz brakujące wartości sprzedaży. W `extra_duplicates` policz dodatkowe wystąpienia klucza, pozostawiając pierwsze nieoznaczone. W `all_duplicates` policz wszystkie wiersze z powtórzonym kluczem, również pierwsze. Niczego nie usuwaj.

In [ ]:
missing_sale = None
extra_duplicates = None
all_duplicates = None
# isna na kolumnie sale; duplicated z listą kolumn klucza i parametrem keep.

In [ ]:
assert missing_sale == 1, "C10.1: jedna sprzedaż jest brakiem; licz isna().sum() na kolumnie sale."
assert extra_duplicates == 1, "C10.1: duplicated([\"item\", \"day\"]) z keep=\"first\" oznacza tylko kolejne wystąpienia."
assert all_duplicates == 2, "C10.1: keep=False oznacza oba wiersze z powtórzonym kluczem."
assert panel.equals(panel_before), "C10.1: raport nie zmienia tabeli."
print("C10.1: brak i powtórzenie są policzone osobno, bez usuwania.")

<details><summary>Podpowiedź 1</summary>

Pusta komórka CSV staje się NaN; isna ją rozpoznaje, porównanie z None nie.

</details>

<details><summary>Podpowiedź 2</summary>

duplicated przyjmuje listę kolumn klucza. Parametr keep decyduje, czy pierwsze wystąpienie też jest oznaczone.

</details>

## Decyzja o danych zapisana przy kodzie

Powtórzony wiersz ma identyczną sprzedaż, więc traktujemy go jako kopię i zostawiamy jedno wystąpienie. Zapisujemy, ilu wierszy dotyczy ta zmiana. Brak sprzedaży plum w dniu 4 pozostaje brakiem; nie wpisujemy zera. Gotowa komórka wykonuje tę decyzję i tworzy `clean`, z której korzystają kolejne zadania.

In [ ]:
clean = panel.drop_duplicates(["item", "day"])
removed_rows = len(panel) - len(clean)
assert removed_rows == 1, "Oczekiwano usunięcia jednej kopii wiersza."
print("Usunięte kopie:", removed_rows, "| wiersze po decyzji:", len(clean))
print("Braki sprzedaży pozostawione:", int(clean["sale"].isna().sum()))

## C10.2 · Jedna wartość na produkt i wartość dla wiersza

Po wykładzie C10.2 w `totals` zapisz sumę sprzedaży według produktu jako Series z `clean`. W `item_means` zapisz dla każdego wiersza `clean` średnią sprzedaży jego produktu, z zachowaniem liczby wierszy. Przewidź, ile wierszy ma każdy wynik i czy brak sprzedaży plum wpływa na średnią.

In [ ]:
totals = None
item_means = None

In [ ]:
assert isinstance(totals, pd.Series) and len(totals) == 3, "C10.2: totals to jedna wartość na produkt, czyli agregacja groupby."
assert {str(k): float(v) for k, v in totals.items()} == {"apple": 22.0, "pear": 63.0, "plum": 12.0}, "C10.2: sumuj clean, nie panel z kopią wiersza; brak nie wchodzi do sumy."
assert isinstance(item_means, pd.Series) and len(item_means) == len(clean), "C10.2: item_means zachowuje liczbę wierszy; to transform, nie agregacja."
expected_means = clean["item"].map({"apple": 4.4, "pear": 10.5, "plum": 2.4}).to_numpy()
assert np.allclose(item_means.to_numpy(dtype=float), expected_means), "C10.2: każdy wiersz dostaje średnią własnego produktu; średnia pomija brak, ale nie zero."
assert float(totals.sum()) == float(clean["sale"].sum()), "C10.2: suma po agregacji ma zgadzać się z sumą przed nią."
print("C10.2: agregacja i transform odpowiadają na dwa różne pytania.")

<details><summary>Podpowiedź 1</summary>

groupby("item")["sale"] z sum daje jedną wartość na grupę.

</details>

<details><summary>Podpowiedź 2</summary>

transform("mean") zwraca wynik grupy przy każdym wierszu. Zero zostaje w średniej, NaN nie.

</details>

## C10.3 · Dołącz opis i sprawdź dopasowania

Po wykładzie C10.3 gotowa komórka wczytuje `data/c-items.csv` z opisem produktów. W `joined` połącz `clean` z `items` po `item`, zachowując wszystkie wiersze sprzedaży, z kontrolą relacji many-to-one i kolumną wskaźnika `_merge`. W `unmatched` policz wiersze sprzedaży bez opisu. Przewidź, ile ich będzie, patrząc na listę produktów w obu tabelach.

In [ ]:
items = pd.read_csv(lab_dir / "data" / "c-items.csv")
print(items)

In [ ]:
joined = None
unmatched = None

In [ ]:
assert isinstance(joined, pd.DataFrame), "C10.3: joined ma być wynikiem merge."
assert len(joined) == len(clean), "C10.3: how=\"left\" zachowuje wszystkie wiersze sprzedaży; nie używaj inner."
assert "group" in joined.columns and "_merge" in joined.columns, "C10.3: dołącz kolumnę group i dodaj indicator=True."
assert float(joined["sale"].sum()) == float(clean["sale"].sum()), "C10.3: złączenie nie może zmienić sumy sprzedaży."
assert unmatched == 6, "C10.3: policz wiersze z _merge równym left_only; jeden produkt nie ma opisu."
print("C10.3: liczba wierszy i suma są zachowane, a brak opisu jest policzony.")

<details><summary>Podpowiedź 1</summary>

Do merge podaj on, how, validate i indicator, tak jak w przykładzie wykładu.

</details>

<details><summary>Podpowiedź 2</summary>

Wiersze bez pary mają w kolumnie _merge wartość left_only. Porównanie całej kolumny z tym napisem daje maskę do zsumowania.

</details>

## Sprawdź rezultat

Sprawdzenia przechodzą po restarcie kernela. Do przemyślenia: co zrobiłby `validate="many_to_one"`, gdyby items miało dwa wiersze apple? Dlaczego liczba wierszy `panel` równa 18 nie dowodziła kompletności danych? Tabela items zawiera produkt bread, którego nie ma w sprzedaży: czy to błąd? Nie musisz spisywać odpowiedzi.